# Introdução à degradação e restauração de imagens

Esta atividade opcional introduz ruído gaussiano, sal-e-pimenta e motion blur, além de comparar filtros básicos da DIP-06 com MSE. O escopo é introdutório: não inclui filtros inversos, Wiener, deconvolução ou restauração em frequência.

## 1. Instalação

No Colab, instale a versão disponível na branch principal.

In [ ]:
%pip install -q "git+https://github.com/tfvieira/dip-2026-2.git"

## 2. Preparação

As degradações retornam `float64` e não fazem clipping implícito. Assim, valores fora da faixa original continuam visíveis para a medição do erro.

In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

from dip_toolkit import download_course_image
from dip_toolkit.modules.image_loader import ImageLoader
from dip_toolkit.modules.image_preprocessor import ImagePreprocessor
from dip_toolkit.modules.image_restorer import ImageRestorer

restorer = ImageRestorer()
filters = ImagePreprocessor()
loader = ImageLoader()

In [ ]:
def show_images(items, columns=3):
    rows = int(np.ceil(len(items) / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4 * rows))
    for axis, (image, title) in zip(axes.ravel(), items, strict=False):
        axis.imshow(image, cmap="gray", vmin=0, vmax=255)
        axis.set_title(title)
        axis.axis("off")
    for axis in axes.ravel()[len(items) :]:
        axis.axis("off")
    figure.tight_layout()

## 3. Degradações sintéticas controladas

Uma seed fixa torna o experimento reproduzível. A imagem combina um gradiente e uma região homogênea para evidenciar ruído e borramento.

In [ ]:
height, width = 160, 200
original = np.tile(np.linspace(30, 220, width), (height, 1)).astype(np.uint8)
original[45:115, 70:130] = 180

gaussian_noisy = restorer.add_gaussian_noise(original, std=20, seed=12)
impulse_noisy = restorer.add_salt_and_pepper_noise(original, 0.04, 0.04, seed=12)
motion_blurred = restorer.apply_motion_blur(
    original, kernel_size=9, direction="horizontal"
)

show_images(
    [
        (original, "Referência"),
        (gaussian_noisy, "Ruído gaussiano"),
        (impulse_noisy, "Sal e pimenta"),
        (motion_blurred, "Motion blur"),
    ],
    columns=2,
)
for name, image in {
    "Gauss": gaussian_noisy,
    "Impulsivo": impulse_noisy,
    "Blur": motion_blurred,
}.items():
    print(f"MSE {name}: {restorer.mse(original, image):.2f}")

## 4. Comparação de filtros

A suavização Gaussiana costuma reduzir ruído gaussiano; a mediana lida melhor com impulsos isolados. Um filtro de média posterior não reverte motion blur, apenas suaviza ainda mais a imagem.

In [ ]:
gaussian_restored = filters.gaussian_filter(gaussian_noisy, kernel_size=5, sigma=1.0)
impulse_restored = filters.median_filter(impulse_noisy, kernel_size=3)
motion_smoothed = filters.mean_filter(motion_blurred, kernel_size=3)

show_images(
    [
        (gaussian_noisy, "Gauss degradada"),
        (gaussian_restored, "Filtro Gauss"),
        (impulse_noisy, "Impulsiva degradada"),
        (impulse_restored, "Filtro de mediana"),
        (motion_blurred, "Motion blur"),
        (motion_smoothed, "Média após blur"),
    ]
)
for name, image in [
    ("Gauss filtrada", gaussian_restored),
    ("Impulsiva filtrada", impulse_restored),
    ("Blur suavizado", motion_smoothed),
]:
    print(f"MSE {name}: {restorer.mse(original, image):.2f}")

## 5. Resposta conhecida do motion blur

Um impulso sintético revela o kernel: a energia de um impulso é distribuída na direção do movimento e o kernel normalizado possui soma um.

In [ ]:
impulse = np.zeros((41, 41), dtype=np.uint8)
impulse[20, 20] = 255
kernel = restorer.motion_blur_kernel(7, "horizontal")
blurred_impulse = restorer.apply_motion_blur(impulse, 7, "horizontal")
show_images([(impulse, "Impulso"), (kernel, "Kernel"), (blurred_impulse, "Resposta")])
print(f"Soma do kernel: {kernel.sum():.1f}")

## 6. Imagem da disciplina

A imagem real é obtida somente por `download_course_image` e visualizada com Matplotlib, sem `cv2.imshow`.

In [ ]:
path = download_course_image("cameraman_original.png")
course_image = loader.load_image(path, flags=cv.IMREAD_GRAYSCALE)
course_image = cv.resize(course_image, (256, 256), interpolation=cv.INTER_AREA)
course_noisy = restorer.add_salt_and_pepper_noise(course_image, 0.03, 0.03, seed=21)
course_restored = filters.median_filter(course_noisy, kernel_size=3)
show_images(
    [
        (course_image, "Referência"),
        (course_noisy, "Sal e pimenta"),
        (course_restored, "Mediana"),
    ]
)
print(f"MSE degradada: {restorer.mse(course_image, course_noisy):.2f}")
print(f"MSE filtrada: {restorer.mse(course_image, course_restored):.2f}")

## Exploração

Altere a seed, o desvio padrão, as probabilidades e o tamanho dos kernels. Compare MSE e aparência visual: uma métrica menor é útil, mas não substitui a inspeção de detalhes e bordas. Como extensão opcional, implemente PSNR a partir do MSE.